### Determining the optimal number of hidden layers and neurons for an Artificial Neural Network (ANN) 
This can be challenging and often requires experimentation. However, there are some guidelines and methods that can help you in making an informed decision:

- Start Simple: Begin with a simple architecture and gradually increase complexity if needed.
- Grid Search/Random Search: Use grid search or random search to try different architectures.
- Cross-Validation: Use cross-validation to evaluate the performance of different architectures.
- Heuristics and Rules of Thumb: Some heuristics and empirical rules can provide starting points, such as:
  -    The number of neurons in the hidden layer should be between the size of the input layer and the size of the output layer.
  -  A common practice is to start with 1-2 hidden layers.

In [17]:
import numpy as np
import pandas as pd

from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from scikeras.wrappers import KerasClassifier

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

In [18]:
import pandas as pd

data = pd.read_csv("./data/Churn_Modelling.csv")
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [19]:
## preprocess the data
data = data.drop(['RowNumber',"CustomerId",'Surname'],axis=1)

from sklearn.preprocessing import LabelEncoder

label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])


## one hot encoder for geography
from sklearn.preprocessing import OneHotEncoder

onehot_encoder_geo = OneHotEncoder(handle_unknown='ignore')
geo_encoded = onehot_encoder_geo.fit_transform(data[['Geography']]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded,columns=onehot_encoder_geo.get_feature_names_out(['Geography']))



## coombine one-hot enoded columns with original data

data= pd.concat([data.drop('Geography',axis=1),geo_encoded_df],axis=1)



## train testand split

X = data.drop("EstimatedSalary",axis=1)
y = data['EstimatedSalary']

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [20]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_trained_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [21]:
# Save the encoders and scaler for later use
import pickle

with open('label_encoder_gender.pkl', 'wb') as file:
    pickle.dump(label_encoder_gender, file)

with open('onehot_encoder_geo.pkl', 'wb') as file:
    pickle.dump(onehot_encoder_geo, file)

with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

In [22]:
## define fun to create a model and try diff parameter(KerasRegressor)

def create_model(neurons=32, layer=1):

    model = Sequential()

    model.add(
        Dense(
            neurons,
            activation='relu',
            input_shape=(X_train.shape[1],)
        )
    )

    for _ in range(layer - 1):
        model.add(
            Dense(
                neurons,
                activation='relu'
            )
        )

    model.add(
        Dense(1)
    )

    model.compile(
        optimizer='adam',
        loss='mean_squared_error',
        metrics=['mean_absolute_error']
    )

    return model

In [23]:
## create a SciKeras regressor
from scikeras.wrappers import KerasRegressor

model = KerasRegressor(
    model=create_model,
    epochs=50,
    batch_size=10,
    verbose=0
)

# Confirm that model-building parameters are routable by GridSearchCV.
model.set_params(model__layer=1, model__neurons=32)

,model,<function cre...002622C7B8BF0>
,batch_size,10
,verbose,0
,epochs,50
,model__layer,1
,model__neurons,32
,build_fn,None
,warm_start,False
,random_state,None
,optimizer,'rmsprop'
,loss,None


In [24]:
from sklearn.model_selection import GridSearchCV

params = {
    'model__neurons': [16, 32, 64, 128],
    'model__layer': [1, 2, 3],
    'epochs': [50, 100]
}

grid = GridSearchCV(
    estimator=model,
    param_grid=params,
    cv=3,
    n_jobs=1
)

grid.fit(X_trained_scaled, y_train)

print('Best Parameters :', grid.best_params_)
print('Best Score      :', grid.best_score_)

c:\Users\DEV_MANISH\anaconda3\envs\deeplearn\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\Users\DEV_MANISH\anaconda3\envs\deeplearn\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\Users\DEV_MANISH\anaconda3\envs\deeplearn\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super

Best Parameters : {'epochs': 50, 'model__layer': 2, 'model__neurons': 128}
Best Score      : -0.007188733951264958
